# Train on a free GPU (Colab / Kaggle)

Why: training on the Mac (MPS) was slow and silently stalled for two epochs. On a T4 GPU this takes about 30-45 min per model.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
Outputs (tables, figures, best model) are zipped at the end so you can download them and commit the small files to GitHub.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 1. Get the code (the repo must contain the latest src/, push from your Mac first)
!rm -rf Hate-Speech-Detection
!git clone https://github.com/aadya197/hate-speech-detection Hate-Speech-Detection
%cd Hate-Speech-Detection
!pip install -q "transformers>=4.45" datasets "accelerate>=1.0" scikit-learn seaborn matplotlib
# Safety check: Colab must have the newest script (it needs --fp16 and saves predictions)
!git log -1 --format="Latest commit: %h %s (%cd)"
!grep -q -- "--fp16" src/train_xlmr.py && grep -q save_predictions src/train_xlmr.py && echo "OK: scripts are up to date" || echo "STALE CLONE: push your latest src/ to GitHub first, then re-run this cell"

In [ ]:
# 2. Build the processed CSVs (downloads HateXplain, needs internet; about 1 minute)
!python -m src.data
!ls -la data/processed

In [ ]:
# 3. Quick sanity check: 1 tiny epoch, makes sure the whole pipeline works on this GPU
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --smoke --train_bs 16 --grad_accum 1 --fp16

## Model A: BERT (main model)
`bert-base-uncased` is the standard model for HateXplain and is English-only, which suits these posts.
Watch the log: `eval_macro_f1` should **change and grow** every epoch. If it prints the WARNING about identical metrics, stop.

In [ ]:
!python -m src.train_xlmr --model_name bert-base-uncased --tag bert --epochs 4 --train_bs 16 --grad_accum 1 --fp16

## Model B: XLM-RoBERTa again, now on GPU (for the comparison table)
Optional but recommended: the Mac run only really trained one epoch, so its 0.56 macro-F1 is not a fair number.
XLM-R is sensitive to the learning rate; 2e-5 with batch size 16 is a safe choice.

Uses a 10% learning-rate warmup (`--warmup_ratio 0.1`). The first epoch's `eval_macro_f1` should be well above 0.5; if it stays near 0.39, or the WARNING about identical metrics appears, stop the run.

In [ ]:
!python -m src.train_xlmr --model_name FacebookAI/xlm-roberta-base --tag xlmr_gpu --epochs 4 --train_bs 16 --grad_accum 1 --fp16 --warmup_ratio 0.1

In [ ]:
# 4. Show the results
import pandas as pd
for tag in ["bert", "xlmr_gpu"]:
    try:
        print(tag); display(pd.read_csv(f"results/tables/{tag}_metrics.csv")); display(pd.read_csv(f"results/tables/{tag}_per_class_test.csv"))
    except FileNotFoundError:
        print(tag, "not run yet")

In [ ]:
# 5. Zip ONLY this run's files (so unzipping on the Mac cannot overwrite your newer local results)
TAG = "xlmr_gpu"      # use "bert" if you re-run BERT
!zip -rq results_gpu.zip results/tables/{TAG}_* results/figures/{TAG}_* models/{TAG}_best
!ls -la results_gpu.zip
from google.colab import files   # on Kaggle: the zip appears under /kaggle/working instead
files.download("results_gpu.zip")

After downloading, unzip into your project folder on the Mac. `models/` is git-ignored (large), but `results/tables` and `results/figures` should be committed.